In [75]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [76]:
from pyspark.sql import SparkSession
spark=SparkSession.builder\
    .appName("DataAnalysis")\
    .getOrCreate()

Port Operations Analytics Engineer


Task 7: Shipment-Port Integration


In [77]:
supply_chain_data=spark.read.parquet("/content/drive/.shortcut-targets-by-id/1TV3Q-5w59wgnF6t4DLUHr0Rjlt9Z2Yxg/data_lake_layer/silver/transformed_data1.parquet")
supply_chain_data.show()

+-----------+-----------+------------+--------------+-------------------+--------------------+----------------+---------------------+--------------+--------------------+---------------------+--------------+--------------------+--------------------------+-----------------+-----------------+------------------+----------------+------------------+----------------+--------------------+--------------+--------------+
|Shipment_ID|Supplier_ID|     Country|  Product_Type|Monthly_Demand_Tons|Shipment_Volume_Tons|Route_Risk_Score|Historical_Delay_Days|Fuel_Price_USD|Political_Risk_Index|Port_Congestion_Index|Inventory_Days|Supplier_Reliability|Alternative_Supplier_Count|Transit_Time_Days|Delay_Probability|Current_Delay_Days|Freight_Cost_USD|Revenue_Impact_USD|Disruption_Event|Freight_Cost_Per_Ton|Delay_Category|Inventory_Risk|
+-----------+-----------+------------+--------------+-------------------+--------------------+----------------+---------------------+--------------+--------------------+---

In [78]:
port_data=spark.read.parquet('/content/drive/.shortcut-targets-by-id/1TV3Q-5w59wgnF6t4DLUHr0Rjlt9Z2Yxg/data_lake_layer/silver/transformed_data2.parquet')
port_data.show()

+-------+--------------+------------+--------------+------------------+--------------------------+------------------------+--------------+------------------------+--------------------------+------------------------+------------------+------------+--------------------+------------------+
|Port_ID|     Port_Name|     Country|     Port_Type|Port_Capacity_Tons|Avg_Vessel_Wait_Time_Hours|Port_Utilization_Percent|Active_Vessels|Weather_Disruption_Score|Labor_Availability_Percent|Monthly_Port_Revenue_USD|Operational_Status|Port_Manager|Last_Inspection_Date|Port_Load_Category|
+-------+--------------+------------+--------------+------------------+--------------------------+------------------------+--------------+------------------------+--------------------------+------------------------+------------------+------------+--------------------+------------------+
|PORT007|Singapore Port|     Germany|  LNG Terminal|            394138|                     43.17|                   53.41|           13

Perform

INNER JOIN
between:
supply_chain_hormuz_crisis_700.csv
port_operations_master.csv
using:
Port_ID


In [79]:
data=supply_chain_data.join(port_data,supply_chain_data['Country']==port_data['Country'],'inner')
data.show()

+-----------+-----------+-------+------------+-------------------+--------------------+----------------+---------------------+--------------+--------------------+---------------------+--------------+--------------------+--------------------------+-----------------+-----------------+------------------+----------------+------------------+----------------+--------------------+--------------+--------------+-------+------------------+-------+--------------+------------------+--------------------------+------------------------+--------------+------------------------+--------------------------+------------------------+------------------+------------+--------------------+------------------+
|Shipment_ID|Supplier_ID|Country|Product_Type|Monthly_Demand_Tons|Shipment_Volume_Tons|Route_Risk_Score|Historical_Delay_Days|Fuel_Price_USD|Political_Risk_Index|Port_Congestion_Index|Inventory_Days|Supplier_Reliability|Alternative_Supplier_Count|Transit_Time_Days|Delay_Probability|Current_Delay_Days|Freigh

Task 8: Port Performance Analysis


Generate:


Report 1


Port-Level KPIs


In [80]:
from pyspark.sql.functions import *

Total Shipment Volume

In [81]:
total_shipment_volume=data.groupBy('Port_ID').agg(sum('Shipment_Volume_Tons').alias('total_shipment_volume'))
total_shipment_volume.show()

+-------+---------------------+
|Port_ID|total_shipment_volume|
+-------+---------------------+
|PORT015|              8124772|
|PORT008|              9497968|
|PORT009|              8848303|
|PORT020|             10195531|
|PORT007|              9888308|
|PORT006|              6973828|
|PORT017|              9367204|
|PORT005|              7179841|
|PORT003|             13486443|
|PORT011|              6144816|
|PORT016|              9483177|
|PORT019|              8070823|
|PORT018|              5888598|
|PORT014|              5709460|
|PORT010|              4239114|
|PORT001|             12930760|
|PORT004|              7417237|
|PORT013|              4636091|
|PORT012|              9299386|
|PORT002|              8483605|
+-------+---------------------+



Average Delay

In [82]:
avg_delay=data.groupBy('Port_ID').agg(avg('Current_Delay_Days').alias('average_delays'))
avg_delay.show()

+-------+------------------+
|Port_ID|    average_delays|
+-------+------------------+
|PORT015|12.152751423149905|
|PORT008| 12.30744071954211|
|PORT009|11.861377506538798|
|PORT020|11.973076923076922|
|PORT007|12.068006182380216|
|PORT006|12.058165548098435|
|PORT017|12.154987633965375|
|PORT005|  12.2518756698821|
|PORT003|11.856082238720731|
|PORT011|11.941785252263907|
|PORT016|12.078577336641853|
|PORT019|12.129682997118156|
|PORT018|11.983957219251337|
|PORT014|12.241333333333333|
|PORT010|11.982206405693951|
|PORT001|12.158273381294965|
|PORT004|12.579001019367992|
|PORT013|12.505728314238953|
|PORT012|12.194750211685013|
|PORT002|12.021316033364226|
+-------+------------------+



Average Freight Cost

In [83]:
avg_freight_cost=data.groupBy('Port_ID').agg(avg('Freight_Cost_USD').alias('average_freight_cost'))

Total Revenue Impact

In [84]:
total_revenue=data.groupBy('Port_ID').agg(sum('Revenue_Impact_USD').alias('total_revenue_impact'))

In [85]:
port_level_kpi = total_shipment_volume \
    .join(avg_delay, "Port_ID") \
    .join(avg_freight_cost, "Port_ID") \
    .join(total_revenue, "Port_ID")

port_level_kpi.show(truncate=False)

+-------+---------------------+------------------+--------------------+--------------------+
|Port_ID|total_shipment_volume|average_delays    |average_freight_cost|total_revenue_impact|
+-------+---------------------+------------------+--------------------+--------------------+
|PORT015|8124772              |12.152751423149905|214388.59107210586  |6.70390265360001E8  |
|PORT008|9497968              |12.30744071954211 |215597.90252657395  |7.81806708970001E8  |
|PORT009|8848303              |11.861377506538798|212471.36741935503  |7.203474394700009E8 |
|PORT020|10195531             |11.973076923076922|215250.86387692305  |8.278795350900017E8 |
|PORT007|9888308              |12.068006182380216|212736.5304945909   |8.112404923299991E8 |
|PORT006|6973828              |12.058165548098435|213529.19153243827  |5.672542896700007E8 |
|PORT017|9367204              |12.154987633965375|213914.4715416322   |7.697727488599998E8 |
|PORT005|7179841              |12.2518756698821  |212286.4722615225   

In [101]:
port_level_kpi.write \
    .mode("append") \
    .parquet("/content/drive/.shortcut-targets-by-id/1TV3Q-5w59wgnF6t4DLUHr0Rjlt9Z2Yxg/data_lake_layer/silver/transformed_data3/port_level_kpi")

Report 2


Port Congestion Analysis


Using:
Port_Load_Category


Calculate:


Average Delay


In [87]:
average_delay=data.groupBy('Port_Load_Category').agg(avg('Current_Delay_Days').alias('average_delay_days'))

Average Freight Cost

In [88]:
average_freight_cost=data.groupBy('Port_Load_Category').agg(avg('Freight_Cost_USD').alias('average_frieght_cost'))

Revenue Impact

In [89]:
revenue_impact=data.groupBy('Port_Load_Category').agg(sum('Revenue_Impact_USD').alias('total_revenue_cost'))

Disruption Percentage

In [90]:
distruption_percentage=data.groupBy('Port_Load_Category').agg(sum("Disruption_Event")*100/count("*").alias("Disruption_Percentage")
    )

In [91]:
port_congestion_report = average_delay \
    .join(average_freight_cost, "Port_Load_Category") \
    .join(revenue_impact, "Port_Load_Category") \
    .join(distruption_percentage, "Port_Load_Category")

port_congestion_report.show(truncate=False)

+------------------+------------------+--------------------+-------------------+-------------------------------------------------------------------+
|Port_Load_Category|average_delay_days|average_frieght_cost|total_revenue_cost |((sum(Disruption_Event) * 100) / count(1) AS Disruption_Percentage)|
+------------------+------------------+--------------------+-------------------+-------------------------------------------------------------------+
|High              |11.988974448722436|214089.36526776233  |3.610563693670024E9|29.59397969898495                                                  |
|Low               |12.117631544243704|214811.6097243819   |4.818356216530001E9|29.94856916787551                                                  |
|Medium            |12.19260628465804 |212408.7112014793   |5.14362507538003E9 |29.192852741836106                                                 |
+------------------+------------------+--------------------+-------------------+--------------------------

In [92]:
port_congestion_report.write \
    .mode("append") \
    .parquet("/content/drive/.shortcut-targets-by-id/1TV3Q-5w59wgnF6t4DLUHr0Rjlt9Z2Yxg/data_lake_layer/silver/transformed_data3/port_congestion_report")

Task 9: Operational Status Impact Analysis

Compare

Normal Ports

Delayed Ports

Critical Ports

For each category calculate:


In [93]:
#Average Delay
avg_delay = data.groupBy("Operational_Status").agg(avg("Current_Delay_Days").alias("Average_Delay"))

In [94]:
# Shipment Volume
shipment_volume = data.groupBy("Operational_Status").agg(sum("Shipment_Volume_Tons").alias("Shipment_Volume")
)

In [95]:
# Revenue Impact
revenue_impact = data.groupBy("Operational_Status").agg(sum("Revenue_Impact_USD").alias("Revenue_Impact")
)

In [96]:
# Freight Cost
freight_cost = data.groupBy("Operational_Status").agg(avg("Freight_Cost_USD").alias("Average_Freight_Cost")
)

In [97]:
operational_status_report = avg_delay \
    .join(shipment_volume, on="Operational_Status", how="inner") \
    .join(revenue_impact, on="Operational_Status", how="inner") \
    .join(freight_cost, on="Operational_Status", how="inner")

operational_status_report.show(truncate=False)

+------------------+------------------+---------------+--------------------+--------------------+
|Operational_Status|Average_Delay     |Shipment_Volume|Revenue_Impact      |Average_Freight_Cost|
+------------------+------------------+---------------+--------------------+--------------------+
|Delayed           |12.132808206631251|42796293       |3.4868402508400264E9|216327.5153801048   |
|Critical          |12.299312714776633|22544897       |1.8506510708400075E9|213712.6860068725   |
|Normal            |12.061028904393162|100524075      |8.235053663899865E9 |212610.89767231708  |
+------------------+------------------+---------------+--------------------+--------------------+



In [98]:
operational_status_report.write \
    .mode("append") \
    .parquet("/content/drive/.shortcut-targets-by-id/1TV3Q-5w59wgnF6t4DLUHr0Rjlt9Z2Yxg/data_lake_layer/silver/transformed_data3/operational_status_report")

Business Questions


In [99]:
# Which operational status creates the largest business impact?

from pyspark.sql.functions import col

operational_status_report.orderBy(col("Revenue_Impact").desc()).show(1, truncate=False)

+------------------+------------------+---------------+-------------------+--------------------+
|Operational_Status|Average_Delay     |Shipment_Volume|Revenue_Impact     |Average_Freight_Cost|
+------------------+------------------+---------------+-------------------+--------------------+
|Normal            |12.061028904393162|100524075      |8.235053663899865E9|212610.89767231708  |
+------------------+------------------+---------------+-------------------+--------------------+
only showing top 1 row


In [100]:
# How much additional revenue impact is generated by Critical Ports?

critical_revenue = revenue_impact.filter(col("Operational_Status")=="Critical")

critical_revenue.show()

+------------------+--------------------+
|Operational_Status|      Revenue_Impact|
+------------------+--------------------+
|          Critical|1.8506510708400075E9|
+------------------+--------------------+

